In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

import altair as alt

from mixres.sim import PoissonEnveloped1D, make_aggregation_matrix
from mixres.models._inference import run_inference_mcmc

from bspline1d_utils import make_bspline_basis
from interval_utils import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_interval_sum_matrix,
)

/Users/shozen/Imperial/0_Research/mixres/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Generation

In [75]:
# Generate the dataset
# left_limits = [0, 0, 5, 8, 10, 15, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90]
# right_limits = [5, 12, 15, 25, 20, 30, 35, 40, 45, 50, 55, 60, 70, 80, 90, 95, 99]

left_limits = [0, 10, 20, 30, 50, 70, 80, 90]
right_limits = [9, 19, 29, 49, 69, 79, 89, 99]

intervals = [
    f"[{left},{right}{']' if right in {5, 15, 25, 30, 40, 50, 60, 70, 80, 90, 99} else ')'}"
    for left, right in zip(left_limits, right_limits)
]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=20, seed=42
).generate(return_true_rates=True)

# Expanded interval dataframe for plotting
df_expanded = expand_interval_dataframe(df_obs, interval_col="obs_interval", x_col="x")

Intervals: ['[0,9)', '[10,19)', '[20,29)', '[30,49)', '[50,69)', '[70,79)', '[80,89)', '[90,99]']


In [76]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(df_true: pd.DataFrame, df_segment: pd.DataFrame) -> alt.Chart:
    source_segment = summarize_interval_segments(df_segment)

    chart_true = (
        alt.Chart(df_true)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"))
    )

    chart_segment = (
        alt.Chart(source_segment)
        .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    return chart_true + chart_segment

In [77]:
plot_data(df_rates, df_expanded)

alt.LayerChart(...)

In [78]:
W = make_aggregation_matrix(left_limits, right_limits)
Phi = make_bspline_basis(x=np.linspace(0, 100, 100), n_basis=40, boundary_ext=0)

In [79]:
beta = np.random.normal(0, 1, size=(40, 1))
f = Phi @ beta

source = pd.DataFrame({"x": np.linspace(0, 100, 100), "f": f.flatten()})
chart = alt.Chart(source).mark_line(color="blue").encode(x="x", y="f")
chart

alt.Chart(...)

In [80]:
def make_fisher_information(
    lambda_: np.ndarray, W: np.ndarray, Phi: np.ndarray
) -> np.ndarray:
    """
    Computes the Fisher information matrix for the Poisson model given the rate,
    aggregation matrix, and B-spline basis.
    """

    inv_eta = 1 / (W @ lambda_ / W.sum(axis=1))
    Dinv_eta = np.diag(inv_eta)
    Dlam = np.diag(lambda_)

    I = Phi.T @ Dlam @ W.T @ Dinv_eta @ W @ Dlam @ Phi
    return I

## Subspace Decomposition

We compare two settings for the Fisher information matrix used to split the spline coefficient space into an identifiable subspace (`PrIm`) and a null space (`PrKer`) via eigendecomposition:

1. **Prior-averaged** — average the Fisher information over many draws of `lambda_` from the prior. Because Fisher information is local (it rotates with `lambda_`), averaging over enough draws collapses the null space toward `{0}` — it answers "is this direction identifiable *somewhere* in the prior," not "is it identifiable *here*."
2. **Empirical point estimate** — evaluate the Fisher information once, at a single data-informed rate estimate derived from the observed counts. This gives a stable, directly interpretable local null space tied to the actual data.

### Setting 1: Prior-Averaged Fisher Information

(current implementation, unchanged)

In [81]:
def forward():
    beta0 = np.random.normal(0, 1)
    beta = np.random.normal(0, 1, size=(40, 1))
    f = beta0 + Phi @ beta
    rate = np.exp(f)

    return rate.flatten()

In [82]:
S = 1000
I_prior = np.zeros((Phi.shape[1], Phi.shape[1]))
for s in range(S):
    lambda_ = forward()
    I_prior += make_fisher_information(lambda_, W, Phi)

I_prior = I_prior / S  # Average over S samples

# I_prior is symmetric PSD by construction (Phi.T @ Dlam @ W.T @ Dinv_eta @ W @ Dlam @ Phi),
# so eigh guarantees real eigenvalues/eigenvectors (unlike eig, which can return
# complex results for a matrix that is only symmetric up to floating-point error).
eigvals_prior, eigvecs_prior = LA.eigh(I_prior)

In [83]:
df_eigvals = pd.DataFrame({"idx": range(len(eigvals_prior)), "eigvals": eigvals_prior})

chart_eigvals = (
    alt.Chart(df_eigvals)
    .mark_line(color="purple")
    .encode(x="idx", y=alt.Y("eigvals:Q", title="Eigenvalues"))
)

chart_eigvals

alt.Chart(...)

In [84]:
# Select non-zero eigenvectors
PrIm_prior = eigvecs_prior[:, eigvals_prior > 12]
PrKer_prior = eigvecs_prior[:, eigvals_prior <= 12]

print(PrIm_prior.shape, PrKer_prior.shape)

(40, 8) (40, 32)


In [85]:
beta_Im_prior = PrIm_prior @ np.random.normal(0, 1, size=(PrIm_prior.shape[1], 1))
beta_Ker_prior = PrKer_prior @ np.random.normal(0, 1, size=(PrKer_prior.shape[1], 1))
f_Im_prior = Phi @ beta_Im_prior
f_Ker_prior = Phi @ beta_Ker_prior

source = pd.DataFrame(
    {
        "x": np.linspace(0, 100, 100),
        "f_Im": f_Im_prior.flatten(),
        "f_Ker": f_Ker_prior.flatten(),
    }
)
chart = alt.Chart(source).mark_line(color="green").encode(x="x", y="f_Im")
chart_Ker = alt.Chart(source).mark_line(color="blue").encode(x="x", y="f_Ker")
chart + chart_Ker

alt.LayerChart(...)

### Setting 2: Empirical Point-Estimate Fisher Information

Instead of averaging over the prior, evaluate the Fisher information once at a single rate estimate derived directly from `df_obs`. Each observation in interval `j` is an i.i.d. draw `y ~ Poisson(eta_j)` (see `PoissonEnveloped1D.generate`), so `eta_hat_j = sum(y) / N_j` is the MLE of the interval's mean rate. This is back-projected to a fine-grid rate estimate `lambda_hat` via the minimum-norm least-squares solution of `interval_avg_op @ lambda = eta_hat` (`interval_avg_op = W / row_widths`, the same interval-averaging operator `make_fisher_information` uses internally) -- this works whether the observed intervals are disjoint or overlapping, and reduces to the obvious piecewise-constant answer in the disjoint case.

In [86]:
df_eta_hat = df_obs.groupby("obs_interval_id")["y"].agg(N="size", y="sum").reset_index()
eta_hat = (df_eta_hat["y"] / df_eta_hat["N"]).to_numpy()

# Back-project eta_hat to the fine grid via the minimum-norm least-squares
# solution of interval_avg_op @ lambda = eta_hat, where interval_avg_op =
# W / row_widths is the exact interval-averaging operator
# make_fisher_information assumes internally. This handles both disjoint
# and overlapping intervals: G (num. observed intervals) is far below N
# (fine-grid size) in this notebook, so the system is underdetermined but
# consistent, and pinv finds an exact solution. For disjoint intervals this
# reduces to the same piecewise-constant assignment as a direct per-point
# average (verified to match to floating-point precision).
#
# Named interval_avg_op rather than the more obvious `A` because `A` is
# already used later in the notebook (Experiment 1 data prep) for an
# unrelated matrix from build_interval_sum_matrix -- reusing `A` here would
# silently shadow that.
interval_avg_op = W / W.sum(axis=1, keepdims=True)
lambda_hat = np.linalg.pinv(interval_avg_op) @ eta_hat
assert np.allclose(interval_avg_op @ lambda_hat, eta_hat)

In [87]:
I_emp = make_fisher_information(lambda_hat, W, Phi)

eigvals_emp, eigvecs_emp = LA.eigh(I_emp)

# A bare `eigvals > 0` split is fragile here: I_emp is built from a single
# point estimate (not averaged over many draws), so its truly-zero
# eigenvalues land at tiny floating-point noise on both sides of zero
# (e.g. +/-1e-14) rather than exactly at it. Use a relative tolerance
# (matching np.linalg.matrix_rank's default convention) instead.
tol = eigvals_emp.max() * len(eigvals_emp) * np.finfo(eigvals_emp.dtype).eps
PrIm_emp = eigvecs_emp[:, eigvals_emp > tol]
PrKer_emp = eigvecs_emp[:, eigvals_emp <= tol]

print(PrIm_emp.shape, PrKer_emp.shape)

(40, 8) (40, 32)


In [88]:
beta_Im_emp = PrIm_emp @ np.random.normal(0, 1, size=(PrIm_emp.shape[1], 1))
beta_Ker_emp = PrKer_emp @ np.random.normal(0, 1, size=(PrKer_emp.shape[1], 1))
f_Im_emp = Phi @ beta_Im_emp
f_Ker_emp = Phi @ beta_Ker_emp

source = pd.DataFrame(
    {
        "x": np.linspace(0, 100, 100),
        "f_Im": f_Im_emp.flatten(),
        "f_Ker": f_Ker_emp.flatten(),
    }
)
chart = alt.Chart(source).mark_line(color="green").encode(x="x", y="f_Im")
chart_Ker = alt.Chart(source).mark_line(color="blue").encode(x="x", y="f_Ker")
chart + chart_Ker

alt.LayerChart(...)

### Setting 3: Oracle Fisher Information

Setting 2 answers "what's locally identifiable given this dataset's rate estimate?" This setting asks the companion question: what's locally identifiable at the *true* rate, if we had oracle knowledge instead of an estimate derived from finite `N_j` observations per interval? Comparing `PrKer_emp` against `PrKer_oracle` isolates estimation noise from the structural non-identifiability of the design (`W`, `Phi`) itself.

We compute `eta_true`, the exact average true rate for each observed interval, from the fine-grid true curve `df_rates["lambda"]` (already plotted everywhere else in this notebook as the ground truth). `lambda_true` is then reconstructed from `eta_true` via the same minimum-norm least-squares back-projection Setting 2 uses for `eta_hat` -> `lambda_hat` (reusing the same operator `interval_avg_op`), so Setting 3 differs from Setting 2 only in using the exact `eta_true` in place of the noisy `eta_hat` -- not in how `Dlam` is built, and both settings now handle overlapping intervals correctly.

In [89]:
lambda_true_fine = df_rates["lambda"].to_numpy()[:100]
eta_true = W @ lambda_true_fine / W.sum(axis=1)

# Back-project via the same minimum-norm least-squares operator
# interval_avg_op as Setting 2 (defined above), so Setting 3 differs from
# Setting 2 only in using the exact eta_true instead of the noisy eta_hat
# -- not in how lambda (and hence Dlam) is constructed.
lambda_true = np.linalg.pinv(interval_avg_op) @ eta_true
assert np.allclose(interval_avg_op @ lambda_true, eta_true)

In [90]:
I_oracle = make_fisher_information(lambda_true, W, Phi)

eigvals_oracle, eigvecs_oracle = LA.eigh(I_oracle)

# Same tolerance-based rank threshold as Setting 2 (a bare `> 0` split is
# fragile for a single-point FIM; the truly-zero eigenvalues land at tiny
# floating-point noise on both sides of zero).
tol = eigvals_oracle.max() * len(eigvals_oracle) * np.finfo(eigvals_oracle.dtype).eps
PrIm_oracle = eigvecs_oracle[:, eigvals_oracle > tol]
PrKer_oracle = eigvecs_oracle[:, eigvals_oracle <= tol]

print(PrIm_oracle.shape, PrKer_oracle.shape)

(40, 8) (40, 32)


In [91]:
beta_Im_oracle = PrIm_oracle @ np.random.normal(0, 1, size=(PrIm_oracle.shape[1], 1))
beta_Ker_oracle = PrKer_oracle @ np.random.normal(0, 1, size=(PrKer_oracle.shape[1], 1))
f_Im_oracle = Phi @ beta_Im_oracle
f_Ker_oracle = Phi @ beta_Ker_oracle

source = pd.DataFrame(
    {
        "x": np.linspace(0, 100, 100),
        "f_Im": f_Im_oracle.flatten(),
        "f_Ker": f_Ker_oracle.flatten(),
    }
)
chart = alt.Chart(source).mark_line(color="green").encode(x="x", y="f_Im")
chart_Ker = alt.Chart(source).mark_line(color="blue").encode(x="x", y="f_Ker")
chart + chart_Ker

alt.LayerChart(...)

In [92]:
# Choose which Fisher-information setting feeds the downstream models.
# PrIm, PrKer = (
#     PrIm_prior,
#     PrKer_prior,
# )  # or: PrIm_emp, PrKer_emp / PrIm_oracle, PrKer_oracle

# PrIm, PrKer = (
#     PrIm_oracle,
#     PrKer_oracle,
# )

PrIm, PrKer = PrIm_emp, PrKer_emp

## Result Plotting Helpers

Shared helpers for the 4 downstream experiments' result plots, so each experiment's own plot and the final side-by-side comparison are built from the same code.

In [93]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior_band(
    source: pd.DataFrame,
    df_true: pd.DataFrame,
    df_segment: pd.DataFrame,
    title: str | None = None,
) -> alt.Chart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    chart = alt.Chart(source).mark_line(color="purple").encode(x="x", y="lam")
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_true, df_segment) + chart + band
    return result.properties(title=title) if title else result

### Experiment 1: Naive B-Spline model

In [94]:
def model_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = Phi.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    beta = numpyro.sample("w", dist.Normal(0, 1).expand([M]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi @ beta))
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [95]:
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

x = np.arange(0, 101)
y = df_model["y"].values
interval_id = df_model["obs_interval_id"].values
N = df_model["N"].values

intervals = df_obs["obs_interval"].unique().tolist()
A = build_interval_sum_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Convert to JAX arrays
y_jax = jnp.array(y)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
W_jax = jnp.array(W)
Phi_jax = jnp.array(Phi)
interval_width_jax = jnp.array(interval_width)

In [96]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_bspline,
    W=W_jax,
    Phi=Phi_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 3225.24it/s, 15 steps of size 2.29e-01. acc. prob=0.89]

Number of divergences: 0
dict_keys(['beta0', 'lam', 'w'])


In [97]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior_band(source_exp1, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 2: Reduced subspace model

In [98]:
def model_reduced_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = PrIm.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([M]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi @ (PrIm @ betaIm)))
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [99]:
PrIm_jax = jnp.array(PrIm)

In [100]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_reduced_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 4830.17it/s, 15 steps of size 3.15e-01. acc. prob=0.91]

Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'lam'])


In [101]:
source_exp2 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "2. Reduced Subspace"
)
plot_posterior_band(source_exp2, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 3: Null-space included model

In [30]:
def model_nullinc_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    MIm = PrIm.shape[1]
    MKer = PrKer.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([MIm]).to_event(1))
    betaKer = numpyro.sample("betaKer", dist.Normal(0, 1).expand([MKer]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic(
        "lam", jnp.exp(beta0 + Phi @ (PrIm @ betaIm + PrKer @ betaKer))
    )
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [31]:
PrKer_jax = jnp.array(PrKer)

In [32]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_nullinc_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    PrKer=PrKer_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 4473.67it/s, 15 steps of size 2.53e-01. acc. prob=0.93]

Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'betaKer', 'lam'])


In [33]:
source_exp3 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "3. Null-Space Included"
)
plot_posterior_band(source_exp3, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 4: Shrinkage-Regularized Null-Space Model

Experiment 2 drops `betaKer` entirely (zero posterior uncertainty in the null space); Experiment 3 gives it the same unshrunk `Normal(0,1)` prior as the identified space (maximal, unregularized uncertainty). This experiment places a **penalized-complexity (PC) shrinkage prior** on `betaKer` instead, so the null space keeps genuine but regularized uncertainty.

A PC prior is chosen over a horseshoe prior: every direction in `PrKer` is equally non-identified by construction (zero Fisher information at the reference `lambda_`), so there's no sparsity structure to recover, which is the horseshoe's whole purpose — and the horseshoe's Half-Cauchy local+global scales are known to produce funnel geometry that's hard for NUTS to sample without extra reparameterization/regularization tricks. A PC prior on a single global scale `tau` (`P(tau > u) = alpha` gives `tau ~ Exponential(-log(alpha)/u)` directly) is simpler and better-behaved.

`pc_u=0.2` (tighter than the identified space's `Normal(0,1)` scale) was chosen after checking divergences directly: `pc_u=1.0` produced 2-6% divergent transitions, because 32 simultaneous null-space directions pushed through the nonlinear `exp()` link can occasionally produce extreme rates — a real irregularity that raising `target_accept_prob` does not fix. Tightening to `pc_u=0.2` eliminated divergences (confirmed across 2 PRNG seeds) while still leaving non-zero posterior uncertainty in the null space.

In [34]:
def model_shrinkage_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
    pc_u: float = 0.2,
    pc_alpha: float = 0.05,
):
    """
    Bayesian B-spline regression model with a penalized-complexity (PC)
    shrinkage prior on the Fisher-information null space, instead of
    excluding it (model_reduced_bspline) or leaving it unshrunk
    (model_nullinc_bspline).

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    PrKer: ArrayLike
        Projection matrix onto the null space of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    pc_u, pc_alpha: float, optional
        PC-prior calibration for the null-space shrinkage scale tau:
        P(tau > pc_u) = pc_alpha.
    """
    MIm = PrIm.shape[1]
    MKer = PrKer.shape[1]

    # PC prior on the null-space shrinkage scale: P(tau > pc_u) = pc_alpha
    # gives tau ~ Exponential(rate), rate = -log(pc_alpha) / pc_u.
    rate = -jnp.log(pc_alpha) / pc_u
    tau = numpyro.sample("tau", dist.Exponential(rate))

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([MIm]).to_event(1))

    # Non-centered parameterization: avoids the funnel geometry a
    # centered betaKer ~ Normal(0, tau) would induce under NUTS.
    betaKer_raw = numpyro.sample(
        "betaKer_raw", dist.Normal(0, 1).expand([MKer]).to_event(1)
    )
    betaKer = numpyro.deterministic("betaKer", betaKer_raw * tau)

    # --- Latent rate ---
    lam = numpyro.deterministic(
        "lam", jnp.exp(beta0 + Phi @ (PrIm @ betaIm + PrKer @ betaKer))
    )
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [35]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_shrinkage_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    PrKer=PrKer_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    y=y_jax,
    pc_u=0.3,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 4249.14it/s, 15 steps of size 2.96e-01. acc. prob=0.91]

Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'betaKer', 'betaKer_raw', 'lam', 'tau'])


In [36]:
source_exp4 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "4. Shrinkage-Regularized"
)
plot_posterior_band(source_exp4, df_rates, df_expanded)

alt.LayerChart(...)

## Comparing All Four Experiments

In [37]:
experiment_order = [
    "1. Naive",
    "2. Reduced Subspace",
    "3. Null-Space Included",
    "4. Shrinkage-Regularized",
]
sources = {
    "1. Naive": source_exp1,
    "2. Reduced Subspace": source_exp2,
    "3. Null-Space Included": source_exp3,
    "4. Shrinkage-Regularized": source_exp4,
}

comparison_charts = [
    plot_posterior_band(sources[name], df_rates, df_expanded, title=name).properties(
        width=200, height=200
    )
    for name in experiment_order
]

alt.hconcat(*comparison_charts).resolve_scale(y="independent")

alt.HConcatChart(...)